# Demo

Do imports.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from ae483.tools import *
from ae483.mytools import *

Load the saved flight data and compare Crazyflie state estimates with the pursuer and runner mocap tracks.

In [ ]:
# Load drone telemetry and both rigid-body tracks from the flight data file.
filename = 'mocap_test.json'
with open(filename, 'r') as f:
    flight_data = json.load(f)

drone = flight_data.get('drone', {})
pursuer_name = 'marker_deck_30'
runner_name = 'marker_deck_16'
bodies = flight_data.get('bodies', {})
pursuer = flight_data.get('mocap_pursuer', bodies.get(pursuer_name, flight_data.get('mocap', {})))
runner = flight_data.get('mocap_runner', bodies.get(runner_name, {}))

print(f"Drone x samples: {len(drone.get('stateEstimate.x', {}).get('data', []))}")
print(f'Pursuer mocap samples: {len(pursuer.get("time", []))}')
print(f'Runner mocap samples: {len(runner.get("time", []))}')

## Overlay Crazyflie telemetry and mocap tracks

The Crazyflie state estimate and Qualisys positions are shown in meters. Since the flight sends the pursuer's Qualisys pose to its estimator, they should share the Qualisys world frame. The plots use each stream's own timestamps; the height plot resets each trace to its first sample.

In [ ]:
def valid_xy(body):
    x = np.asarray(body.get('x', []), dtype=float)
    y = np.asarray(body.get('y', []), dtype=float)
    t = np.asarray(body.get('time', []), dtype=float)
    n = min(len(x), len(y), len(t))
    valid = np.isfinite(x[:n]) & np.isfinite(y[:n]) & np.isfinite(t[:n])
    return t[:n][valid], x[:n][valid], y[:n][valid]

def valid_drone_xy(drone_data):
    x_log = drone_data.get('stateEstimate.x', {})
    y_log = drone_data.get('stateEstimate.y', {})
    n = min(len(x_log.get('time', [])), len(x_log.get('data', [])),
            len(y_log.get('time', [])), len(y_log.get('data', [])))
    t = np.asarray(x_log.get('time', [])[:n], dtype=float)
    x = np.asarray(x_log.get('data', [])[:n], dtype=float)
    y = np.asarray(y_log.get('data', [])[:n], dtype=float)
    valid = np.isfinite(t) & np.isfinite(x) & np.isfinite(y)
    return t[valid], x[valid], y[valid]

t_pur, x_pur, y_pur = valid_xy(pursuer)
t_run, x_run, y_run = valid_xy(runner)
t_cf, x_cf, y_cf = valid_drone_xy(drone)

if not len(x_cf):
    print('No Crazyflie stateEstimate.x/y samples found in this file.')
if not len(x_pur) or not len(x_run):
    print('One or both mocap tracks have no valid samples. Check use_mocap, body names, and the flight data file.')

Plot the Crazyflie position estimate and both mocap tracks in the x-y plane.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), tight_layout=True)
if len(x_cf):
    ax.plot(x_cf, y_cf, label='Crazyflie state estimate')
    ax.scatter(x_cf[0], y_cf[0], marker='o', s=45, label='Crazyflie start')
if len(x_pur):
    ax.plot(x_pur, y_pur, '--', label=f'pursuer mocap ({pursuer_name})')
if len(x_run):
    ax.plot(x_run, y_run, ':', label=f'runner mocap ({runner_name})')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Crazyflie estimate and mocap trajectories')
ax.set_aspect('equal', adjustable='datalim')
ax.grid(True)
ax.legend()
plt.show()

Plot the Crazyflie's estimated height and both mocap heights over their respective elapsed times.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), tight_layout=True)
traces = [
    (pursuer.get('time', []), pursuer.get('z', []), f'pursuer mocap ({pursuer_name})', '--'),
    (runner.get('time', []), runner.get('z', []), f'runner mocap ({runner_name})', ':'),
]
cf_z = drone.get('stateEstimate.z', {})
traces.append((cf_z.get('time', []), cf_z.get('data', []), 'Crazyflie state estimate', '-'))
for time_values, z_values, label, style in traces:
    time_s = np.asarray(time_values, dtype=float)
    z = np.asarray(z_values, dtype=float)
    n = min(len(time_s), len(z))
    valid = np.isfinite(time_s[:n]) & np.isfinite(z[:n])
    if np.any(valid):
        time_valid = time_s[:n][valid]
        ax.plot(time_valid - time_valid[0], z[:n][valid], style, label=label)
ax.set_xlabel('time since each trace first valid sample (s)')
ax.set_ylabel('z (m)')
ax.grid(True)
ax.legend()
plt.show()

### Reflection

Review the reflection you put in your Lab 1 notebook. **Modify the text in this cell** to answer the following three questions with at least one sentence each:
* What was your biggest struggle this week? Was it the same or different from last week?

The largest struggle was keeping straight which frames each variable was related to and how they related to each other.

* What was the most important thing you learned this week?

getting more comfortable working with the varying frames and transforming between them.

* How does what you learned this week relate to what you learned last week?

The use of the mocap provided a separtate set of data to validate position for the same flight path used in lab 01 